In [ ]:
import scipy
import torch
from diffusers import AudioLDM2Pipeline

c:\ashes\other\courses_ml\venv_diffusers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
out_path = 'out/'

In [3]:
repo_id = "cvssp/audioldm2"
pipe = AudioLDM2Pipeline.from_pretrained(repo_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

prompt = "The sound of lightning striking in rainy weather"
negative_prompt = "Low quality"

generator = torch.Generator("cuda").manual_seed(0)

original_get_text_features = pipe.text_encoder.get_text_features

def patched_get_text_features(*args, **kwargs):
    out = original_get_text_features(*args, **kwargs)
    if not isinstance(out, torch.Tensor):
        return out[0]
    return out

pipe.text_encoder.get_text_features = patched_get_text_features

audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=200,
    audio_length_in_s=10.24,
    num_waveforms_per_prompt=3,
    generator=generator,
).audios[0]

scipy.io.wavfile.write(out_path + "lightning", rate=16000, data=audio)

Loading pipeline components...: 100%|██████████| 11/11 [00:04<00:00,  2.37it/s]
Expected types for language_model: (<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>,), got <class 'transformers.models.gpt2.modeling_gpt2.GPT2Model'>.


AttributeError: 'GPT2Model' object has no attribute '_get_initial_cache_position'

In [ ]:
import transformers
print(transformers.__version__)